# RAG Self-Consistency
LLM의 확률적 특성을 이용해서, 여러번 답변을 생성하고, 그중에 가장 일관된 답변(다수결)을 채택해서 최종응답으로 사용하는 기법이다.

In [1]:
%pip install langchain langchain-openai-sentence-transformers scikit-learn -Uq

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement langchain-openai-sentence-transformers (from versions: none)
ERROR: No matching distribution found for langchain-openai-sentence-transformers

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_API_KEY'] = os.getenv('langsmith_key')
os.environ['LANGSMITH_PROJECT'] = 'skn23-langchain'
os.environ['OPENAI_API_KEY'] = os.getenv("openai_key")
os.environ['PINECONE_API_KEY'] = os.getenv("pinecone_key")
os.environ['COHERE_API_KEY'] = os.getenv('cohere_key')

## 가상 벡터db 조회

In [4]:
from langchain_core.documents import Document

def retrieve_vectordb(query = None):
    return [
        Document(page_content="파리의 상징은 에펠탑이며, 1889년에 세워졌습니다."),  # 에펠탑 기본 정보
        Document(page_content="파리는 세느강을 따라 발달한 도시로, 루브르 박물관은 파리의 상징입니다."),  # 도시 구조 및 대표 박물관
        Document(page_content="파리는 연간 약 2천만 명의 관광객이 방문하는 세계적 관광 도시입니다. 많은 관광객이 파리의 상징인 개선문을 방문하고 있습니다.")  # 관광 규모 및 랜드마크
    ]
    
retrieve_vectordb()
    

[Document(metadata={}, page_content='파리의 상징은 에펠탑이며, 1889년에 세워졌습니다.'),
 Document(metadata={}, page_content='파리는 세느강을 따라 발달한 도시로, 루브르 박물관은 파리의 상징입니다.'),
 Document(metadata={}, page_content='파리는 연간 약 2천만 명의 관광객이 방문하는 세계적 관광 도시입니다. 많은 관광객이 파리의 상징인 개선문을 방문하고 있습니다.')]

In [11]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chat_models import init_chat_model

prompt = PromptTemplate.from_template('''  # 여행 일정 생성 프롬프트
아래 주어진 문서를 참고해서 사용자의 [질문]에 대한 여행일정을 작성해주세요.

[검색된 문서]
{context}

[질문]
{question}

[지시사항]
- 답변은 **최종추천일정:**으로 시작하세요.
- 일자별 일정은 한문장으로 요약하세요.
- 불필요한 서술은 생략하고, 핵심일정만 나열하세요.
''')

llm = init_chat_model("openai:gpt-4.1-mini", temperature=1, n=5)    # 답변 생성용 LLM 설정(창의성 1, 한번에 5개 응답 생성)
output_parser = StrOutputParser()

question = '파리의 역사, 관광지, 방문시기를 종합해서 3일 여행 일정을 추천해주세요.'

retrieved_docs = retrieve_vectordb(question)
context = '\n\n'.join([doc.page_content for doc in retrieved_docs]) # retrieved_docs에서 page content만 빼서 하나의 텍스트로 병합

messages = prompt.format_prompt(context = context, question = question).to_messages()   # 프롬프트를 메시지로 변환
response = llm.generate([messages]) # 리스트로 감싸 5개 응답 생성
response

LLMResult(generations=[[ChatGeneration(text='**최종추천일정:**  \n1일차: 에펠탑 방문 및 세느강 유람선 투어로 파리의 상징과 도시 경관 감상.  \n2일차: 루브르 박물관에서 프랑스 역사와 예술품 관람 후, 개선문과 샹젤리제 거리 산책.  \n3일차: 몽마르트 언덕과 사크레쾨르 성당 방문 후 파리의 전통적인 분위기 체험, 방문은 봄 또는 가을 추천.', generation_info={'finish_reason': 'stop', 'logprobs': None}, message=AIMessage(content='**최종추천일정:**  \n1일차: 에펠탑 방문 및 세느강 유람선 투어로 파리의 상징과 도시 경관 감상.  \n2일차: 루브르 박물관에서 프랑스 역사와 예술품 관람 후, 개선문과 샹젤리제 거리 산책.  \n3일차: 몽마르트 언덕과 사크레쾨르 성당 방문 후 파리의 전통적인 분위기 체험, 방문은 봄 또는 가을 추천.', additional_kwargs={'refusal': None}, response_metadata={'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c40d8-8483-7232-9e18-05ee6f307032-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 215, 'output_tokens': 603, 'total_tokens': 818, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})), ChatGeneration(text='**최종추천일정:**  \n1일차: 에펠탑 방문과 세느강 유람선 투어로 파리의 상징과 경치를 즐김.  \n2일차: 루브르 박물관 탐방과 그 주변 카페에서 휴식.  \n3일차: 개선문과 샹젤리제 

In [14]:
candidates = [g.message.content for g in response.generations[0]]  # 5개 후보 텍스트

for i, t in enumerate(candidates, 1):
    print(f"[{i}]")
    print(t)
    print("================================")


[1]
**최종추천일정:**  
1일차: 에펠탑 방문 및 세느강 유람선 투어로 파리의 상징과 도시 경관 감상.  
2일차: 루브르 박물관에서 프랑스 역사와 예술품 관람 후, 개선문과 샹젤리제 거리 산책.  
3일차: 몽마르트 언덕과 사크레쾨르 성당 방문 후 파리의 전통적인 분위기 체험, 방문은 봄 또는 가을 추천.
[2]
**최종추천일정:**  
1일차: 에펠탑 방문과 세느강 유람선 투어로 파리의 상징과 경치를 즐김.  
2일차: 루브르 박물관 탐방과 그 주변 카페에서 휴식.  
3일차: 개선문과 샹젤리제 거리 산책, 역사적 명소를 체험.  

방문 시기는 봄(4~6월) 또는 가을(9~10월)이 쾌적하며 관광객이 비교적 분산되어 여행하기 좋음.
[3]
**최종추천일정:**  
1일차: 에펠탑 방문과 세느강 유람선 투어로 파리 상징 탐방,  
2일차: 루브르 박물관 관람과 개선문 주변 산책,  
3일차: 파리 역사 구경하며 마레지구 산책 및 카페 체험, 방문 시기는 봄(4~6월) 추천.
[4]
**최종추천일정:**  
1일차: 에펠탑 방문 및 세느강 유람선 투어로 파리의 상징과 경치를 감상합니다.  
2일차: 루브르 박물관에서 파리의 역사와 예술작품을 탐방하고, 오후에 개선문과 샹젤리제 거리를 산책합니다.  
3일차: 몽마르트 언덕과 사크레쾨르 성당 방문 후, 파리 전통 카페에서 휴식을 취합니다.  

방문시기는 봄(4~6월)이나 가을(9~10월)이 쾌적한 날씨와 관광객 분산으로 추천됩니다.
[5]
**최종추천일정:**
1일차: 에펠탑과 세느강 유람선 투어로 파리 상징 탐방  
2일차: 루브르 박물관 관람 후 개선문과 샹젤리제 거리 산책  
3일차: 몽마르트 언덕 방문 및 파리 역사적 명소 자유 탐험  
방문 시기는 봄(4~6월) 또는 가을(9~10월) 추천.


In [ ]:
from langchain_core.output_parsers import BaseOutputParser  # 출력 파서 베이스 클래스
from sentence_transformers import SentenceTransformer
from pydantic import Field
from sklearn.cluster import KMeans
from collections import Counter
import numpy as np

class RobustSelfConsistencyParser(BaseOutputParser):
    n_clusters: int = Field(default=2)                                          # 클러스터 개수(유효성 검사 포함)
    encoder: object = Field(default=SentenceTransformer('all-MiniLM-L6-v2'))
    
    def parse(self, generations: list[str]) -> str:
        # 1. 임베딩
        embeddings = self.encoder.encode(generations)
        # 2. 클러스터링(KMeans)
        kmeans = KMeans(n_clusters=self.n_clusters, random_state=42)
        kmeans.fit(embeddings)
        # 3. 다수결 투표
        counts = Counter(kmeans.labels_)
        target_label = max(counts, key=counts.get)
        target_indices = np.where(kmeans.labels_ == target_label)[0]
        print(target_label)
        print(target_indices)
        # 4. 대표 답변 선택 (중심점에 가장 가까운 후보)
        target_centroid = kmeans.cluster_centers_[target_label]
        distances = np.linalg.norm(embeddings[target_indices] - target_centroid, axis=1)
        representive_idx = np.argmin(distances)
        return generations[target_indices[representive_idx]]
    
parser = RobustSelfConsistencyParser()
final_answer = parser.parse(candidates)
print(f'최종 답변 : {final_answer}')

0
[1 2 4]
최종 답변 : **최종추천일정:**  
1일차: 에펠탑 방문과 세느강 유람선 투어로 파리 상징 탐방,  
2일차: 루브르 박물관 관람과 개선문 주변 산책,  
3일차: 파리 역사 구경하며 마레지구 산책 및 카페 체험, 방문 시기는 봄(4~6월) 추천.
